# detach-clone-snapshot — ex2: .detach() shares storage vs .detach().clone() copies — contrast on in-place mutation

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `detach-clone-snapshot`. Running the final beacon cell reports progress against the `PyTorch: detach + clone snapshot` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: detach + clone snapshot` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-clone-snapshot`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-clone-snapshot"
DD_SUBTOPIC = "PyTorch: detach + clone snapshot"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `.detach()` shares storage; `.detach().clone()` is independent

Ex1 built the full `.detach().clone()` snapshot. The deepening move is to CONTRAST the two operations — show that `.detach()` ALONE shares storage with the source.

```python
x = t.tensor([1.0, 2.0, 3.0], requires_grad=True)
shared = x.detach()             # severs graph; storage is the SAME
snap = x.detach().clone()       # severs graph AND copies storage
x.data[0] = 99.0
shared[0]  # → 99.0   (storage shared)
snap[0]    # → 1.0    (independent copy)
```

**Why `.detach()` shares storage by design.** It's a graph operation, not a memory operation. The whole point is 'view of the same tensor, minus the autograd graph link'. If you wanted a new buffer, you'd ask for one — that's what `.clone()` is for.

**Why this is a common bug.** `model.weight.detach()` looks like a snapshot — but if anyone later writes into `model.weight.data`, the 'snapshot' updates too. The mental model 'detach() = copy' is wrong. The right model: 'detach() = read-only graph cut; clone() = make a real copy'.

**Sharing flag — `.data_ptr()`.** Two tensors share storage iff they have the same `.data_ptr()` (and overlapping byte ranges). This is the load-bearing assertion for this drill.

### Exercise 2 — .detach() shares storage vs .detach().clone() copies — contrast on in-place mutation

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the storage-sharing contrast between `x.detach()` and `x.detach().clone()` by comparing `data_ptr()` and observing which 'snapshot' tracks an in-place write to the source.
> Keywords: detach, clone, storage, data_ptr, view
> ```

**KCs targeted:** `detach-shares-data-ptr`, `clone-allocates-fresh-storage`

Implement `ex2_detach_vs_clone(x)`.

Given a leaf tensor `x` with `requires_grad=True`:
1. Take `shared = x.detach()` (graph-cut only).
2. Take `snap = x.detach().clone()` (graph-cut + fresh storage).
3. Mutate the FIRST element of `x` in place via `x.data[0] = 99.0`. (Cannot directly assign to a `requires_grad=True` leaf, but `.data` bypass works.)
4. Return:
   ```
   {
     'shared_data_ptr_equals_x': bool,         # True
     'snap_data_ptr_equals_x': bool,           # False
     'shared_first_after_mutation': float,     # 99.0
     'snap_first_after_mutation': float,       # original x[0]
     'shared_requires_grad': bool,             # False
     'snap_requires_grad': bool,               # False
     'x_requires_grad': bool,                  # True (unchanged)
   }
   ```

Constraints:
- Use `.data_ptr()` (not `is` or `==`) to compare storage identity.
- Cast comparison values to plain Python `float`.
- Do NOT call `.backward()` — there's no graph to walk; this is a pure storage/graph drill.

In [ ]:
def ex2_detach_vs_clone(x):
    shared = x.detach()
    snap = x.detach().clone()
    x.data[0] = 99.0
    return {
        'shared_data_ptr_equals_x': shared.data_ptr() == x.data_ptr(),
        'snap_data_ptr_equals_x': snap.data_ptr() == x.data_ptr(),
        'shared_first_after_mutation': float(shared[0]),
        'snap_first_after_mutation': float(snap[0]),
        'shared_requires_grad': bool(shared.requires_grad),
        'snap_requires_grad': bool(snap.requires_grad),
        'x_requires_grad': bool(x.requires_grad),
    }


<details><summary>Solution</summary>

```python
def ex2_detach_vs_clone(x):
    shared = x.detach()
    snap = x.detach().clone()
    x.data[0] = 99.0
    return {
        'shared_data_ptr_equals_x': shared.data_ptr() == x.data_ptr(),
        'snap_data_ptr_equals_x': snap.data_ptr() == x.data_ptr(),
        'shared_first_after_mutation': float(shared[0]),
        'snap_first_after_mutation': float(snap[0]),
        'shared_requires_grad': bool(shared.requires_grad),
        'snap_requires_grad': bool(snap.requires_grad),
        'x_requires_grad': bool(x.requires_grad),
    }
```

**`.detach()` is a graph operation only.** It returns a tensor that shares the same storage but is disconnected from autograd. Cheap (no copy), but means any in-place mutation to the source is visible through the detached view.

**`.clone()` allocates a new buffer.** The result has its own `data_ptr()` and is independent. `.detach().clone()` does both: cut the graph AND copy storage. This is the safe snapshot for 'I want to remember this value'.

**`x.data[0] = 99.0` bypasses autograd.** A direct `x[0] = 99.0` on a leaf with `requires_grad=True` raises a RuntimeError because it would corrupt the graph. Mutating via `.data` is the documented escape hatch — but the SAME mutation is exactly why detach-without-clone is dangerous in practice.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()